In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

import utils.data_processing_bronze_table
import utils.data_processing_silver_loan
import utils.data_processing_silver_attributes
import utils.data_processing_silver_financials
import utils.data_processing_silver_clickstream
import utils.data_processing_gold_table



## Setup config

In [3]:
# set path for Hadoop
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["PATH"] + ";C:\\hadoop\\bin"

### set up pyspark session

In [4]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

### set up config

In [5]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2025-11-01"

In [6]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
dates_str_lst

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01',
 '2025-01-01',
 '2025-02-01',
 '2025-03-01',
 '2025-04-01',
 '2025-05-01',
 '2025-06-01',
 '2025-07-01',
 '2025-08-01',
 '2025-09-01',
 '2025-10-01',
 '2025-11-01']

## Build Bronze Table

In [7]:
# create bronze datalake
bronze_loan_directory = "datamart/bronze/loan/"
bronze_clickstream_directory = "datamart/bronze/clickstream/"
bronze_attributes_directory = "datamart/bronze/attributes/"
bronze_financials_directory = "datamart/bronze/financials/"

for d in [bronze_loan_directory, bronze_clickstream_directory, 
          bronze_attributes_directory, bronze_financials_directory]:
    if not os.path.exists(d):
        os.makedirs(d)

In [8]:
# run bronze backfill for all tables
bronze_tables = [
    ("lms_loan_daily", bronze_loan_directory),
    ("feature_clickstream", bronze_clickstream_directory),
    ("features_attributes", bronze_attributes_directory),
    ("features_financials", bronze_financials_directory),
]

for data_file_name, directory in bronze_tables:
    print(f"\n=== Processing {data_file_name} ===")
    for date_str in dates_str_lst:
        utils.data_processing_bronze_table.process_bronze_table(
            date_str, directory, spark, data_file_name
        )


=== Processing lms_loan_daily ===
2023-01-01 | lms_loan_daily | row count: 530
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_01_01.csv
2023-02-01 | lms_loan_daily | row count: 1031
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_02_01.csv
2023-03-01 | lms_loan_daily | row count: 1537
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_03_01.csv
2023-04-01 | lms_loan_daily | row count: 2047
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_04_01.csv
2023-05-01 | lms_loan_daily | row count: 2568
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_05_01.csv
2023-06-01 | lms_loan_daily | row count: 3085
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_06_01.csv
2023-07-01 | lms_loan_daily | row count: 3556
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_07_01.csv
2023-08-01 | lms_loan_daily | row count: 4037
saved to: datamart/bronze/loan/bronze_lms_loan_daily_2023_08_01.csv
2023-09-01 | lms_loan_daily | row count: 4491
saved to

In [9]:
# inspect one partition per table
for data_file_name, directory in bronze_tables:
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    print(f"{data_file_name}: {len(files)} partitions saved")

lms_loan_daily: 35 partitions saved
feature_clickstream: 35 partitions saved
features_attributes: 35 partitions saved
features_financials: 35 partitions saved


In [10]:
# inspect bronze output, one sample partition per table
sample_date = "2023_05_01"

for data_file_name, directory in bronze_tables:
    partition_name = f"bronze_{data_file_name}_{sample_date}.csv"
    filepath = os.path.join(directory, partition_name)
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    print(f"\n=== {data_file_name} | {sample_date} ===")
    print(f"Row count: {df.count()}, Columns: {len(df.columns)}")
    df.show(3)


=== lms_loan_daily | 2023_05_01 ===
Row count: 2568, Columns: 11
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|             loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|snapshot_date|
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+
|CUS_0x1000_2023_0...| CUS_0x1000|     2023-05-01|    10|              0|   10000|    0.0|     0.0|        0.0|10000.0|   2023-05-01|
|CUS_0x1037_2023_0...| CUS_0x1037|     2023-01-01|    10|              4|   10000| 1000.0|  1000.0|        0.0| 6000.0|   2023-05-01|
|CUS_0x1056_2023_0...| CUS_0x1056|     2023-04-01|    10|              1|   10000| 1000.0|     0.0|     1000.0|10000.0|   2023-05-01|
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+-----------

## Build Silver Table

In [11]:
# create folder for silver
silver_loan_directory = "datamart/silver/loan/"
if not os.path.exists(silver_loan_directory):
    os.makedirs(silver_loan_directory)

In [12]:
# Run Silver backfill for separated files. Here is Silver loan table.

for date_str in dates_str_lst:
    utils.data_processing_silver_loan.process_silver_loan(
        date_str, bronze_loan_directory, silver_loan_directory, spark
    )

loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_01_01.csv row count: 530
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_01_01.parquet
loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_02_01.csv row count: 1031
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_02_01.parquet
loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_03_01.csv row count: 1537
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_03_01.parquet
loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_04_01.csv row count: 2047
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_04_01.parquet
loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_05_01.csv row count: 2568
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_05_01.parquet
loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_06_01.csv row count: 3085
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_06_01.parquet
loaded from: datamart/bronze/loan/bronze_lms_lo

In [13]:
# Check Silver loan
date_str = "2023-05-01"
utils.data_processing_silver_loan.process_silver_loan(date_str, bronze_loan_directory, silver_loan_directory, spark).toPandas()

loaded from: datamart/bronze/loan/bronze_lms_loan_daily_2023_05_01.csv row count: 2568
saved to: datamart/silver/loan/silver_lms_loan_daily_2023_05_01.parquet


,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date,mob,installments_missed,first_missed_date,dpd
0,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,0,10000.0,0.0,0.0,0.0,10000.0,2023-05-01,0,0,None,0
1,CUS_0x1037_2023_01_01,CUS_0x1037,2023-01-01,10,4,10000.0,1000.0,1000.0,0.0,6000.0,2023-05-01,4,0,None,0
2,CUS_0x1056_2023_04_01,CUS_0x1056,2023-04-01,10,1,10000.0,1000.0,0.0,1000.0,10000.0,2023-05-01,1,1,2023-04-01,30
3,CUS_0x1069_2023_01_01,CUS_0x1069,2023-01-01,10,4,10000.0,1000.0,1000.0,0.0,6000.0,2023-05-01,4,0,None,0
4,CUS_0x108a_2023_05_01,CUS_0x108a,2023-05-01,10,0,10000.0,0.0,0.0,0.0,10000.0,2023-05-01,0,0,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2563,CUS_0xfaf_2023_04_01,CUS_0xfaf,2023-04-01,10,1,10000.0,1000.0,1000.0,0.0,9000.0,2023-05-01,1,0,None,0
2564,CUS_0xfb6_2023_04_01,CUS_0xfb6,2023-04-01,10,1,10000.0,1000.0,1000.0,0.0,9000.0,2023-05-01,1,0,None,0
2565,CUS_0xfc9_2023_01_01,CUS_0xfc9,2023-01-01,10,4,10000.0,1000.0,0.0,3000.0,9000.0,2023-05-01,4,3,2023-02-01,89
2566,CUS_0xfcb_2023_04_01,CUS_0xfcb,2023-04-01,10,1,10000.0,1000.0,1000.0,0.0,9000.0,2023-05-01,1,0,None,0


In [14]:
silver_attributes_directory = "datamart/silver/attributes/"
if not os.path.exists(silver_attributes_directory):
    os.makedirs(silver_attributes_directory)


In [15]:
# Silver backfill for attributes table.
for date_str in dates_str_lst:
    utils.data_processing_silver_attributes.process_silver_attributes(
        date_str, bronze_attributes_directory, silver_attributes_directory, spark
    )

loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_01_01.csv | row count: 530
saved to: datamart/silver/attributes/silver_features_attributes_2023_01_01.parquet
loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_02_01.csv | row count: 501
saved to: datamart/silver/attributes/silver_features_attributes_2023_02_01.parquet
loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_03_01.csv | row count: 506
saved to: datamart/silver/attributes/silver_features_attributes_2023_03_01.parquet
loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_04_01.csv | row count: 510
saved to: datamart/silver/attributes/silver_features_attributes_2023_04_01.parquet
loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_05_01.csv | row count: 521
saved to: datamart/silver/attributes/silver_features_attributes_2023_05_01.parquet
loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_06_01.csv | row co

In [16]:
# Check Silver attributes
date_str = "2023-05-01"
utils.data_processing_silver_attributes.process_silver_attributes(date_str, bronze_attributes_directory, silver_attributes_directory, spark).toPandas()

loaded from: datamart/bronze/attributes/bronze_features_attributes_2023_05_01.csv | row count: 521
saved to: datamart/silver/attributes/silver_features_attributes_2023_05_01.parquet


,Customer_ID,Age,Occupation,snapshot_date
0,CUS_0x1000,18.0,Lawyer,2023-05-01
1,CUS_0x108a,38.0,Journalist,2023-05-01
2,CUS_0x10f9,54.0,None,2023-05-01
3,CUS_0x1119,36.0,Entrepreneur,2023-05-01
4,CUS_0x1192,NaN,Media_Manager,2023-05-01
...,...,...,...,...
516,CUS_0xe2f,30.0,Developer,2023-05-01
517,CUS_0xe36,29.0,Mechanic,2023-05-01
518,CUS_0xf01,35.0,Developer,2023-05-01
519,CUS_0xf45,28.0,Mechanic,2023-05-01


In [17]:
silver_financials_directory = "datamart/silver/financials/"
if not os.path.exists(silver_financials_directory):
    os.makedirs(silver_financials_directory)

In [18]:
# Run Silver backfill for financials table

for date_str in dates_str_lst:
    utils.data_processing_silver_financials.process_silver_financials(
        date_str, bronze_financials_directory, silver_financials_directory, spark
    )

loaded from: datamart/bronze/financials/bronze_features_financials_2023_01_01.csv | row count: 530
saved to: datamart/silver/financials/silver_features_financials_2023_01_01.parquet
loaded from: datamart/bronze/financials/bronze_features_financials_2023_02_01.csv | row count: 501
saved to: datamart/silver/financials/silver_features_financials_2023_02_01.parquet
loaded from: datamart/bronze/financials/bronze_features_financials_2023_03_01.csv | row count: 506
saved to: datamart/silver/financials/silver_features_financials_2023_03_01.parquet
loaded from: datamart/bronze/financials/bronze_features_financials_2023_04_01.csv | row count: 510
saved to: datamart/silver/financials/silver_features_financials_2023_04_01.parquet
loaded from: datamart/bronze/financials/bronze_features_financials_2023_05_01.csv | row count: 521
saved to: datamart/silver/financials/silver_features_financials_2023_05_01.parquet
loaded from: datamart/bronze/financials/bronze_features_financials_2023_06_01.csv | row co

In [19]:
# Check Silver fianancials
date_str = "2023-05-01"
utils.data_processing_silver_financials.process_silver_financials(date_str, bronze_financials_directory, silver_financials_directory, spark).toPandas()

loaded from: datamart/bronze/financials/bronze_features_financials_2023_05_01.csv | row count: 521
saved to: datamart/silver/financials/silver_features_financials_2023_05_01.parquet


,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,snapshot_date,spending_level,payment_value_level
0,CUS_0x1000,30625.939453,2706.161621,6.0,5.0,27.0,2.0,57,26.0,1.630000,...,1562.910034,30.077190,129,Yes,42.941090,77.314278,400.360809,2023-05-01,1.0,1.0
1,CUS_0x108a,36982.359375,2554.813721,7.0,9.0,21.0,9.0,15,15.0,24.610001,...,4882.120117,32.002995,93,Yes,454.419556,275.807465,84.708839,2023-05-01,0.0,1.0
2,CUS_0x10f9,150131.687500,11102.135742,5.0,1.0,4.0,0.0,8,0.0,7.490000,...,1138.359985,45.255806,383,No,NaN,180.759140,1328.938232,2023-05-01,1.0,1.0
3,CUS_0x1119,56301.898438,4593.825195,9.0,5.0,26.0,2.0,15,18.0,14.730000,...,2161.229980,37.381496,174,Yes,50.119347,310.844452,378.418701,2023-05-01,0.0,1.0
4,CUS_0x1192,16319.375000,1518.947876,7.0,3.0,1.0,2.0,16,13.0,7.670000,...,1275.319946,31.404446,285,None,16.291128,112.339867,303.263794,2023-05-01,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,CUS_0xe2f,64404.160156,4861.145020,6.0,8.0,21.0,6.0,42,19.0,9.050000,...,2075.580078,32.071163,244,Yes,NaN,482.333160,136.916595,2023-05-01,0.0,1.0
517,CUS_0xe36,46264.710938,3857.392578,8.0,9.0,27.0,NaN,51,12.0,3.340000,...,3025.100098,31.956572,150,Yes,249.029404,127.588844,269.121002,2023-05-01,1.0,0.0
518,CUS_0xf01,30256.849609,2226.404053,0.0,5.0,5.0,3.0,21,2.0,0.730000,...,1483.229980,28.165590,245,No,62.105156,157.190964,273.344299,2023-05-01,0.0,2.0
519,CUS_0xf45,32057.300781,2606.441650,9.0,8.0,16.0,7.0,45,25.0,1.400000,...,1327.260010,38.598793,124,Yes,164.859421,207.577087,178.207657,2023-05-01,0.0,0.0


In [20]:
silver_clickstream_directory = "datamart/silver/clickstream/"
if not os.path.exists(silver_clickstream_directory):
    os.makedirs(silver_clickstream_directory)

In [21]:
# Run Silver backfill for clickstream table

for date_str in dates_str_lst:
    utils.data_processing_silver_clickstream.process_silver_clickstream(
        date_str, bronze_clickstream_directory, silver_clickstream_directory, spark
    )

loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_01_01.csv | row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_01_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_02_01.csv | row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_02_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_03_01.csv | row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_03_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_04_01.csv | row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_04_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_05_01.csv | row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_05_01.parquet
loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_06

In [22]:
# Check Silver clickstream
date_str = "2023-05-01"
utils.data_processing_silver_clickstream.process_silver_clickstream(date_str, bronze_clickstream_directory, silver_clickstream_directory, spark).toPandas()

loaded from: datamart/bronze/clickstream/bronze_feature_clickstream_2023_05_01.csv | row count: 8974
saved to: datamart/silver/clickstream/silver_feature_clickstream_2023_05_01.parquet


,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
0,203,75,223,88,188,164,82,18,195,50,...,4,-41,184,150,212,139,1,46,CUS_0x1037,2023-05-01
1,-4,72,151,79,170,181,20,335,320,191,...,249,197,209,-19,202,61,43,124,CUS_0x1069,2023-05-01
2,150,165,96,21,-45,109,196,162,188,233,...,-177,250,97,129,-152,120,185,162,CUS_0x114a,2023-05-01
3,-19,269,78,70,126,-93,195,-31,99,131,...,122,123,362,60,84,36,70,220,CUS_0x1184,2023-05-01
4,46,69,19,-30,-44,-41,194,94,45,-37,...,362,28,100,108,91,-217,78,73,CUS_0x1297,2023-05-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,65,133,210,97,181,191,-116,185,141,100,...,43,103,235,138,78,200,145,153,CUS_0xdf6,2023-05-01
8970,149,-28,150,93,165,-48,-46,244,123,49,...,261,146,307,68,-49,77,248,107,CUS_0xe23,2023-05-01
8971,136,218,-65,-105,-156,124,71,77,130,-1,...,340,204,126,238,73,1,192,-40,CUS_0xe4e,2023-05-01
8972,85,395,67,41,41,103,128,47,152,108,...,58,155,101,15,-140,90,41,190,CUS_0xedd,2023-05-01


## Build Gold table for labels/features

In [23]:
# create gold layer
gold_label_store_directory = "datamart/gold/label_store/"
gold_feature_store_directory = "datamart/gold/feature_store/"

if not os.path.exists(gold_label_store_directory):
    os.makedirs(gold_label_store_directory)

if not os.path.exists(gold_feature_store_directory):
    os.makedirs(gold_feature_store_directory)

In [24]:
# Run gold LABEL store backfill
for date_str in dates_str_lst:
    utils.data_processing_gold_table.process_labels_gold_table(
        date_str, 
        silver_loan_directory, 
        gold_label_store_directory, 
        spark, 
        dpd=30, 
        mob=6
    )

loaded from: datamart/silver/loan/silver_lms_loan_daily_2023_01_01.parquet row count: 530
saved to: datamart/gold/label_store/gold_label_store_2023_01_01.parquet
loaded from: datamart/silver/loan/silver_lms_loan_daily_2023_02_01.parquet row count: 1031
saved to: datamart/gold/label_store/gold_label_store_2023_02_01.parquet
loaded from: datamart/silver/loan/silver_lms_loan_daily_2023_03_01.parquet row count: 1537
saved to: datamart/gold/label_store/gold_label_store_2023_03_01.parquet
loaded from: datamart/silver/loan/silver_lms_loan_daily_2023_04_01.parquet row count: 2047
saved to: datamart/gold/label_store/gold_label_store_2023_04_01.parquet
loaded from: datamart/silver/loan/silver_lms_loan_daily_2023_05_01.parquet row count: 2568
saved to: datamart/gold/label_store/gold_label_store_2023_05_01.parquet
loaded from: datamart/silver/loan/silver_lms_loan_daily_2023_06_01.parquet row count: 3085
saved to: datamart/gold/label_store/gold_label_store_2023_06_01.parquet
loaded from: datamart/s

In [25]:
# Run gold FEATURE store backfill
for date_str in dates_str_lst:
    utils.data_processing_gold_table.process_gold_feature_store(
        date_str,
        silver_loan_directory,
        silver_attributes_directory,
        silver_financials_directory,
        silver_clickstream_directory,
        gold_feature_store_directory,
        spark,
    )

2023-01-01 | new loans: 530
saved to: datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet | features: 530 rows x 106 cols
2023-02-01 | new loans: 501
saved to: datamart/gold/feature_store/gold_feature_store_2023_02_01.parquet | features: 501 rows x 106 cols
2023-03-01 | new loans: 506
saved to: datamart/gold/feature_store/gold_feature_store_2023_03_01.parquet | features: 506 rows x 106 cols
2023-04-01 | new loans: 510
saved to: datamart/gold/feature_store/gold_feature_store_2023_04_01.parquet | features: 510 rows x 106 cols
2023-05-01 | new loans: 521
saved to: datamart/gold/feature_store/gold_feature_store_2023_05_01.parquet | features: 521 rows x 106 cols
2023-06-01 | new loans: 517
saved to: datamart/gold/feature_store/gold_feature_store_2023_06_01.parquet | features: 517 rows x 106 cols
2023-07-01 | new loans: 471
saved to: datamart/gold/feature_store/gold_feature_store_2023_07_01.parquet | features: 471 rows x 106 cols
2023-08-01 | new loans: 481
saved to: datamart/g

Py4JJavaError: An error occurred while calling o46520.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 11 in stage 2505.0 failed 1 times, most recent failure: Lost task 11.0 in stage 2505.0 (TID 3429) (DESKTOP-QGJJJLI executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)

In [ ]:
# Test gold feature store - single partition
date_str = "2023-05-01"
df_test = utils.data_processing_gold_table.process_gold_feature_store(
    date_str,
    silver_loan_directory,
    silver_attributes_directory,
    silver_financials_directory,
    silver_clickstream_directory,
    gold_feature_store_directory,
    spark,
)
df_test.toPandas()

2023-05-01 | new loans: 521
saved to: datamart/gold/feature_store/gold_feature_store_2023_05_01.parquet | features: 521 rows x 106 cols


,Customer_ID,loan_id,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,...,fe_19_mean,fe_19_std,fe_19_min,fe_19_max,fe_20_mean,fe_20_std,fe_20_min,fe_20_max,has_clickstream,snapshot_date
0,CUS_0x1000,CUS_0x1000_2023_05_01,18.0,7.0,30625.939453,2706.161621,6.0,5.0,27.0,2.0,...,80.8,83.902920,4,200,111.2,80.334924,31,234,1,2023-05-01
1,CUS_0x108a,CUS_0x108a_2023_05_01,38.0,6.0,36982.359375,2554.813721,7.0,9.0,21.0,9.0,...,61.2,70.563447,17,184,177.4,190.700813,-88,416,1,2023-05-01
2,CUS_0x10f9,CUS_0x10f9_2023_05_01,54.0,NaN,150131.687500,11102.135742,5.0,1.0,4.0,0.0,...,104.0,82.240501,-7,221,52.0,84.080319,-45,147,1,2023-05-01
3,CUS_0x1119,CUS_0x1119_2023_05_01,36.0,5.0,56301.898438,4593.825195,9.0,5.0,26.0,2.0,...,43.0,77.961529,-64,125,86.2,50.251368,24,161,1,2023-05-01
4,CUS_0x1192,CUS_0x1192_2023_05_01,NaN,10.0,16319.375000,1518.947876,7.0,3.0,1.0,2.0,...,144.8,104.506938,-16,246,75.4,82.832964,9,211,1,2023-05-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,CUS_0xe2f,CUS_0xe2f_2023_05_01,30.0,2.0,64404.160156,4861.145020,6.0,8.0,21.0,6.0,...,94.4,139.528850,-60,276,43.8,72.675305,-59,108,1,2023-05-01
517,CUS_0xe36,CUS_0xe36_2023_05_01,29.0,9.0,46264.710938,3857.392578,8.0,9.0,27.0,NaN,...,126.4,75.118573,5,197,35.6,54.270618,-21,114,1,2023-05-01
518,CUS_0xf01,CUS_0xf01_2023_05_01,35.0,2.0,30256.849609,2226.404053,0.0,5.0,5.0,3.0,...,64.0,103.348924,-45,208,96.6,78.446160,-9,192,1,2023-05-01
519,CUS_0xf45,CUS_0xf45_2023_05_01,28.0,9.0,32057.300781,2606.441650,9.0,8.0,16.0,7.0,...,110.4,89.410849,-26,175,58.0,43.531598,-15,98,1,2023-05-01


In [ ]:
df_test.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Annual_Income: float (nullable = true)
 |-- Monthly_Inhand_Salary: float (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: integer (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: integer (nullable = true)
 |-- Changed_Credit_Limit: float (nullable = true)
 |-- Num_Credit_Inquiries: integer (nullable = true)
 |-- Credit_Mix: integer (nullable = true)
 |-- Outstanding_Debt: float (nullable = true)
 |-- Credit_Utilization_Ratio: float (nullable = true)
 |-- Credit_History_Age: integer (nullable = true)
 |-- Payment_of_Min_Amount: integer (nullable = true)
 |-- Total_EMI_per_month: float (nullable = true)
 |-- Amount_invested_monthly: float (nu

In [ ]:
# Inspect full gold feature store
folder_path = gold_label_store_directory
files_list = [folder_path + os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df_fs = spark.read.option("header", "true").parquet(*files_list)
print("row_count:", df_fs.count())
df_fs.show()

row_count: 12500
+-----------+--------------------+----+----------+-------------+---------------------+-----------------+---------------+-------------+-----------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+------------------+---------------------+-------------------+-----------------------+---------------+--------------+-------------------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+---------+------------------+--------+--------+----------+------------------+---------+---------+----------+------------------+---------+---------+----------+------------------+---------+

In [ ]:
df_label = utils.data_processing_gold_table.process_labels_gold_table(
    "2024-05-01", silver_loan_directory, gold_label_store_directory, spark, dpd=30, mob=6
)
df_label.toPandas()

loaded from: datamart/silver/loan/silver_lms_loan_daily_2024_05_01.parquet row count: 5391
saved to: datamart/gold/label_store/gold_label_store_2024_05_01.parquet


,loan_id,Customer_ID,label,label_def,snapshot_date
0,CUS_0x1011_2023_11_01,CUS_0x1011,0,30dpd_6mob,2024-05-01
1,CUS_0x1018_2023_11_01,CUS_0x1018,1,30dpd_6mob,2024-05-01
2,CUS_0x1041_2023_11_01,CUS_0x1041,0,30dpd_6mob,2024-05-01
3,CUS_0x105b_2023_11_01,CUS_0x105b,0,30dpd_6mob,2024-05-01
4,CUS_0x107c_2023_11_01,CUS_0x107c,0,30dpd_6mob,2024-05-01
...,...,...,...,...,...
486,CUS_0xe57_2023_11_01,CUS_0xe57,1,30dpd_6mob,2024-05-01
487,CUS_0xeda_2023_11_01,CUS_0xeda,1,30dpd_6mob,2024-05-01
488,CUS_0xf19_2023_11_01,CUS_0xf19,0,30dpd_6mob,2024-05-01
489,CUS_0xf3e_2023_11_01,CUS_0xf3e,1,30dpd_6mob,2024-05-01


In [ ]:
folder_path = gold_label_store_directory
files_list = [folder_path + os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df_ls = spark.read.option("header", "true").parquet(*files_list)
print("row_count:", df_ls.count())
df_ls.show()

row_count: 12500
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x10ac_2024_0...| CUS_0x10ac|    0|30dpd_6mob|   2025-02-01|
|CUS_0x10c5_2024_0...| CUS_0x10c5|    1|30dpd_6mob|   2025-02-01|
|CUS_0x1145_2024_0...| CUS_0x1145|    1|30dpd_6mob|   2025-02-01|
|CUS_0x11ac_2024_0...| CUS_0x11ac|    0|30dpd_6mob|   2025-02-01|
|CUS_0x122c_2024_0...| CUS_0x122c|    0|30dpd_6mob|   2025-02-01|
|CUS_0x1274_2024_0...| CUS_0x1274|    1|30dpd_6mob|   2025-02-01|
|CUS_0x1288_2024_0...| CUS_0x1288|    1|30dpd_6mob|   2025-02-01|
|CUS_0x12cc_2024_0...| CUS_0x12cc|    1|30dpd_6mob|   2025-02-01|
|CUS_0x1338_2024_0...| CUS_0x1338|    0|30dpd_6mob|   2025-02-01|
|CUS_0x1370_2024_0...| CUS_0x1370|    1|30dpd_6mob|   2025-02-01|
|CUS_0x1378_2024_0...| CUS_0x1378|    1|30dpd_6mob|   2025-02-01|
|CUS_0x139b_2024_0...| CUS_0x139b|    0|30dpd_6mob|   2025-

In [ ]:
df_ls.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- label_def: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

